# 04. LifeMate AI Agent — Huấn Luyện QLoRA & Tạo Bài Nộp trên Kaggle (GPU T4 x2 / P100)

> **Mục tiêu:** Fine-tune mô hình **Qwen3-4B-Instruct-2507** trên 5.086 lượt kịch bản hội thoại chuẩn ChatML của Zalo AI Challenge 2026.
> **Thời gian huấn luyện:** ~15 - 25 phút (thay vì bị nghẽn 12 tiếng do cấu hình đa GPU ảo hóa).
> **Kết quả đầu ra:** Adapter LoRA tối ưu + File nộp bài `submission_public_test.jsonl` chuẩn hợp lệ 7 tầng.


## 1. Kiểm tra Card Đồ Họa (GPU P100 Status)

In [ ]:
!nvidia-smi

## 2. Cài đặt các thư viện AI cần thiết

In [ ]:
!pip uninstall -y torchao
!pip install -q "transformers>=4.45" accelerate peft bitsandbytes datasets sentencepiece huggingface_hub


## 3. Clone Mã Nguồn vào `/kaggle/working/LifeMateAgent`

In [ ]:
import os
from pathlib import Path

WORKSPACE = Path("/kaggle/working/LifeMateAgent")
if not WORKSPACE.exists():
    !git clone https://github.com/Platypus27-coder/zalo-ai-challenge.git {WORKSPACE}

%cd {WORKSPACE}
!git checkout main
!git pull origin main

## 4. Chuẩn bị Dữ Liệu Train & Test
Hệ thống sẽ tự động quét thư mục `/kaggle/input/` để phát hiện dữ liệu (kể cả đã giải nén hay file `.zip`).

In [ ]:
import os, shutil
from pathlib import Path

DATA_DIR = WORKSPACE / "train+public_test"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 1. Tìm và liên kết thư mục train/ và public_test/ nếu Kaggle đã giải nén sẵn
if Path("/kaggle/input").exists():
    found_train = list(Path("/kaggle/input").rglob("train.jsonl"))
    found_test = list(Path("/kaggle/input").rglob("public_test.jsonl"))

    if found_train and not (DATA_DIR / "train" / "train.jsonl").exists():
        train_src = found_train[0].parent
        target_train = DATA_DIR / "train"
        try:
            os.symlink(str(train_src), str(target_train))
            print(f"🔗 Đã liên kết thư mục train từ: {train_src}")
        except Exception:
            !cp -r "{train_src}" "{target_train}"

    if found_test and not (DATA_DIR / "public_test" / "public_test.jsonl").exists():
        test_src = found_test[0].parent
        target_test = DATA_DIR / "public_test"
        try:
            os.symlink(str(test_src), str(target_test))
            print(f"🔗 Đã liên kết thư mục public_test từ: {test_src}")
        except Exception:
            !cp -r "{test_src}" "{target_test}"

# 2. Nếu dữ liệu dạng file zip trong /kaggle/input, tự động giải nén
if not (DATA_DIR / "train" / "train.jsonl").exists() and Path("/kaggle/input").exists():
    zips = list(Path("/kaggle/input").rglob("*.zip"))
    for z in zips:
        if "train" in z.name.lower() or "lifemate" in z.name.lower() or "test" in z.name.lower():
            print(f"📦 Đang giải nén zip từ: {z}")
            !unzip -q -o "{z}" -d {WORKSPACE}/
            break

# Kiểm tra tính sẵn sàng
train_ok = (DATA_DIR / "train" / "train.jsonl").exists()
test_ok = (DATA_DIR / "public_test" / "public_test.jsonl").exists()

if train_ok and test_ok:
    print("\n✅ Dữ liệu train (1.545 kịch bản) và public_test (634 mẫu) đã sẵn sàng 100%!")
elif train_ok:
    print("\n✅ Dữ liệu train (1.545 kịch bản) đã sẵn sàng!")
else:
    print("\n⚠️ Chưa thấy train.jsonl trong /kaggle/input!")
    print("👉 Cách nạp: Bấm '+ Add Input' ở góc trên bên phải Kaggle để upload file train+public_test.zip")


## 5. Bóc tách 1.545 kịch bản thành 5.086 lượt huấn luyện SFT (ChatML)
Mỗi lượt huấn luyện mô phỏng Teacher-Forcing có mặt đầy đủ các thông tin ngữ cảnh, danh mục tool và phản hồi trợ lý.

In [ ]:
!python scripts/prepare_sft_data.py

## 6. Huấn luyện LoRA Đầy Đủ & Hội Tụ (~30-35 phút trên GPU)
> 🎯 **Chiến lược Huấn luyện 1.500 Bước (~2.4 Epochs):**
> - Trước đây, do lỗi chia GPU trên T4x2 khiến mỗi bước mất 72 giây (5.5 tiếng cho 250 bước, model chưa hội tụ loss = 0.58 dẫn đến điểm 9.37).
> - Nay với `CUDA_VISIBLE_DEVICES=0` và tự động tối ưu GPU đơn, tốc độ đạt **~1.0s – 1.2s / bước**!
> - Huấn luyện **1.500 bước** (~2.4 epochs) chỉ mất **~25 – 35 phút**, giúp hàm Loss giảm sâu xuống **~0.10**, model học trọn vẹn toàn bộ 5.086 lượt hội thoại và điều kiện dừng `DONE` chính xác.
> - Tự động lưu checkpoint mỗi 300 bước.

In [ ]:
ADAPTER_OUT = "/kaggle/working/qwen3_4b_lora"

# Chạy huấn luyện chuẩn 1.500 bước (~2.4 epochs, hoàn thành trong ~25-35 phút):
!CUDA_VISIBLE_DEVICES=0 python scripts/train_lora.py \
    --max-steps 1500 \
    --lr 2e-4 \
    --batch-size 1 \
    --grad-accum 8 \
    --save-steps 300 \
    --output-dir {ADAPTER_OUT}


## 7. Kiểm tra Adapter & Nén file zip cho Kaggle Output

In [ ]:
import shutil
from pathlib import Path

adapter_dir = Path(ADAPTER_OUT)
has_final = (adapter_dir / "adapter_model.safetensors").exists() or (adapter_dir / "adapter_model.bin").exists()

# Nếu chưa có bản lưu cuối nhưng có checkpoint trung gian, tự động copy checkpoint mới nhất
if not has_final and (adapter_dir / "checkpoints").exists():
    ckpts = sorted(list((adapter_dir / "checkpoints").glob("checkpoint-*")), key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else 0)
    if ckpts:
        latest = ckpts[-1]
        print(f"⚠️ Chưa có final export, đang copy checkpoint mới nhất: {latest.name}...")
        for item in latest.iterdir():
            if item.is_file():
                shutil.copy2(item, adapter_dir / item.name)
        has_final = (adapter_dir / "adapter_model.safetensors").exists() or (adapter_dir / "adapter_model.bin").exists()

if has_final:
    print(f"🎉 Huấn luyện thành công! Trọng số lưu tại: {adapter_dir}")
    # Nén zip để tải về trực tiếp từ tab Output trên thanh bên phải Kaggle
    zip_path = Path("/kaggle/working/qwen3_4b_lora_adapter")
    shutil.make_archive(str(zip_path), "zip", str(adapter_dir))
    print(f"📦 File zip tải về: {zip_path}.zip")
else:
    print("❌ Chưa tìm thấy adapter weights. Vui lòng kiểm tra lại tiến trình train.")


## 8. Đẩy LoRA Adapter lên Hugging Face Hub (Khuyên dùng - Private Repo)
Adapter LoRA chỉ nặng **~50MB**. Đưa lên Hugging Face Hub giúp lưu trữ vĩnh viễn và có thể nạp thẳng ở bất kỳ đâu.

### 🔑 Cách lấy Hugging Face Token miễn phí (30 giây):
1. Truy cập: https://huggingface.co/settings/tokens
2. Nhấp nút **Create new token** $\rightarrow$ Chọn Type: **Write** (bắt buộc chọn Write để có quyền tải model lên) $\rightarrow$ Đặt tên bất kỳ $\rightarrow$ Nhấp **Create token**.
3. Sao chép chuỗi mã bắt đầu bằng `hf_...` và dán vào biến `HF_TOKEN` ở ô code bên dưới.

In [ ]:
# ==============================================================================
# ĐIỀN THÔNG TIN TÀI KHOẢN HUGGING FACE CỦA BẠN (TÙY CHỌN - ĐỂ LƯU KHI TẮT MÁY):
# ==============================================================================
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"   # <-- DÁN WRITE TOKEN CỦA BẠN VÀO ĐÂY
HF_REPO_ID = "your_username/lifemate-qwen3-4b-lora" # <-- ĐỔI 'your_username' THÀNH TÊN TÀI KHOẢN CỦA BẠN

if HF_TOKEN and not HF_TOKEN.startswith("hf_xxx") and not "your_username" in HF_REPO_ID:
    print(f"🚀 Đang đẩy Adapter lên Hugging Face Hub: {HF_REPO_ID}...")
    !python scripts/push_to_hub.py \
        --adapter-dir {ADAPTER_OUT} \
        --repo-id {HF_REPO_ID} \
        --token {HF_TOKEN}
else:
    print("ℹ️ Chưa điền HF_TOKEN hợp lệ. Bỏ qua bước đẩy lên Hugging Face.")
    print("📦 Trọng số và file nộp bài vẫn được lưu tự động vĩnh viễn trong mục Output của Kaggle!")


## 9. Chạy Tạo Bài Nộp Cho 634 Mẫu Public Test
Chạy suy luận toàn bộ 634 mẫu với Não bộ LoRA mới và bộ lọc Guardrails chống lặp/chống ảo giác.

In [ ]:
SUBMISSION_OUT = "/kaggle/working/submission_public_test.jsonl"

!CUDA_VISIBLE_DEVICES=0 python scripts/generate_submission.py \
    --split public_test \
    --asr-mode precomputed \
    --agent-mode transformers \
    --model-name Qwen/Qwen3-4B-Instruct-2507 \
    --adapter-path {ADAPTER_OUT} \
    --output {SUBMISSION_OUT} \
    --max-steps 5 \
    --verbose


## 10. Kiểm tra Hợp Chuẩn 7 Lớp & Tải File Nộp Bài
Chạy kiểm tra linter chính thức để đảm bảo 100% hợp lệ trước khi gửi BTC.

In [ ]:
!python scripts/lint_submission.py {SUBMISSION_OUT}

# Nếu đã điền HF_TOKEN, tự động đẩy luôn file nộp bài lên Hugging Face để tải về từ điện thoại/máy tính
if 'HF_TOKEN' in locals() and HF_TOKEN and not HF_TOKEN.startswith("hf_xxx") and not "your_username" in HF_REPO_ID:
    print(f"🚀 Đang đẩy file nộp bài lên Hugging Face: {HF_REPO_ID}...")
    !python scripts/push_to_hub.py \
        --adapter-dir {ADAPTER_OUT} \
        --repo-id {HF_REPO_ID} \
        --submission-file {SUBMISSION_OUT} \
        --token {HF_TOKEN}

print("\n" + "=" * 60)
print("🎉 CHÚC MỪNG BẠN! FILE NỘP BÀI ĐÃ SẴN SÀNG!")
print("👉 Cách 1 (Kaggle): Nhìn sang thanh bên phải (Kaggle sidebar) -> Tab 'Output' -> Tải 'submission_public_test.jsonl'")
print("👉 Cách 2 (Hugging Face): Mở link repo HF của bạn để tải file nộp bài bất cứ lúc nào!")
print("=" * 60)
